In [1]:
from nba_api.stats.endpoints import MatchupsRollup

rollup = MatchupsRollup(
    season="2025-26",
    season_type_playoffs="Regular Season",
    per_mode_simple="Totals",
).matchups_rollup.get_data_frame()

print("Rows:", len(rollup))
print("Columns:", rollup.columns.tolist())

display(rollup["POSITION"].value_counts(dropna=False))

display(
    rollup[
        [
            "DEF_PLAYER_NAME",
            "POSITION",
            "PERCENT_OF_TIME",
            "PARTIAL_POSS",
            "MATCHUP_MIN",
        ]
    ].head(15)
)


Rows: 2326
Columns: ['SEASON_ID', 'POSITION', 'PERCENT_OF_TIME', 'DEF_PLAYER_ID', 'DEF_PLAYER_NAME', 'GP', 'MATCHUP_MIN', 'PARTIAL_POSS', 'PLAYER_PTS', 'TEAM_PTS', 'MATCHUP_AST', 'MATCHUP_TOV', 'MATCHUP_BLK', 'MATCHUP_FGM', 'MATCHUP_FGA', 'MATCHUP_FG_PCT', 'MATCHUP_FG3M', 'MATCHUP_FG3A', 'MATCHUP_FG3_PCT', 'MATCHUP_FTM', 'MATCHUP_FTA', 'SFL']


POSITION
G        582
F        582
TOTAL    582
C        580
Name: count, dtype: int64

,DEF_PLAYER_NAME,POSITION,PERCENT_OF_TIME,PARTIAL_POSS,MATCHUP_MIN
0,LeBron James,G,0.420201,1616.7,315:04
1,LeBron James,F,0.475616,1823.0,356:37
2,LeBron James,C,0.104183,421.6,78:07
3,LeBron James,TOTAL,1.000000,3861.3,749:48
4,Chris Paul,G,0.680626,275.4,53:38
5,Chris Paul,C,0.051607,30.2,4:04
6,Chris Paul,F,0.267766,117.9,21:06
7,Chris Paul,TOTAL,1.000000,423.5,78:48
8,Kyle Lowry,TOTAL,1.000000,238.3,43:48
9,Kyle Lowry,C,0.062785,13.8,2:45


In [3]:

from pathlib import Path
from nba_archetypes.config import RAW_DATA_DIR

raw_path = Path(
    RAW_DATA_DIR / "matchups_rollup.parquet"
)

raw_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

rollup.to_parquet(raw_path, index=False)


In [4]:

import numpy as np
import pandas as pd


POSITIONS = ["G", "F", "C"]


def calculate_matchup_diversity(
    rollup,
    min_possessions=200,
):
    """Measure diversity of recorded G/F/C matchups."""

    required = {
        "DEF_PLAYER_ID",
        "POSITION",
        "PARTIAL_POSS",
    }

    if not required.issubset(rollup.columns):
        raise ValueError("Missing matchup columns.")

    data = rollup.copy()

    data["POSITION"] = (
        data["POSITION"].astype(str).str.strip().str.upper()
    )

    data["PARTIAL_POSS"] = pd.to_numeric(
        data["PARTIAL_POSS"],
        errors="raise",
    )

    valid_positions = set(POSITIONS + ["TOTAL"])

    if not data["POSITION"].isin(valid_positions).all():
        raise ValueError("Unexpected position category.")

    if (
        data["PARTIAL_POSS"].isna().any()
        or (data["PARTIAL_POSS"] < 0).any()
    ):
        raise ValueError("Invalid matchup possessions.")

    if data.duplicated(
        ["DEF_PLAYER_ID", "POSITION"]
    ).any():
        raise ValueError("Duplicate defender-position rows.")

    # Keep the three offensive position categories.
    position_rows = data[
        data["POSITION"].isin(POSITIONS)
    ]

    counts = position_rows.pivot(
        index="DEF_PLAYER_ID",
        columns="POSITION",
        values="PARTIAL_POSS",
    )

    counts = counts.reindex(
        columns=POSITIONS
    ).fillna(0)

    # Total recorded matchup possessions.
    volume = counts.sum(axis=1)

    # Share of matchups against each position.
    shares = counts.div(
        volume.replace(0, np.nan),
        axis=0,
    ).fillna(0)

    p = shares.to_numpy()

    # Handle zero shares without taking log(0).
    safe_p = np.where(p > 0, p, 1.0)

    entropy = -(p * np.log(safe_p)).sum(axis=1)
    diversity = 100 * entropy / np.log(3)

    result = shares.rename(
        columns={
            "G": "MATCHUP_SHARE_G",
            "F": "MATCHUP_SHARE_F",
            "C": "MATCHUP_SHARE_C",
        }
    ).copy()

    result["MATCHUP_POSSESSIONS"] = volume
    result["MATCHUP_DIVERSITY_RAW"] = diversity

    # Compare G/F/C totals against the endpoint's
    # supplied TOTAL row as a data-quality check.
    total_rows = data[
        data["POSITION"] == "TOTAL"
    ].set_index("DEF_PLAYER_ID")["PARTIAL_POSS"]

    result["ROLLUP_TOTAL_POSS"] = total_rows

    result["ROLLUP_COVERAGE"] = (
        result["MATCHUP_POSSESSIONS"]
        / result["ROLLUP_TOTAL_POSS"].replace(0, np.nan)
    )

    # Provisional quality thresholds.
    result["VALID_FOR_MODEL"] = (
        (result["MATCHUP_POSSESSIONS"] >= min_possessions)
        & result["ROLLUP_COVERAGE"].between(0.95, 1.05)
    )

    # Missing means insufficient or unverified evidence,
    # NOT zero defensive versatility.
    result["POSITIONAL_MATCHUP_DIVERSITY"] = (
        result["MATCHUP_DIVERSITY_RAW"].where(
            result["VALID_FOR_MODEL"]
        )
    )

    return result.reset_index()


In [5]:
from nba_archetypes.feature_engineering import (
    calculate_matchup_diversity,
)

diversity = calculate_matchup_diversity(rollup)

print("Total defenders:", len(diversity))
print("Valid defenders:", diversity["VALID_FOR_MODEL"].sum())

print("\nCoverage statistics:")
display(
    diversity["ROLLUP_COVERAGE"].describe()
)

print("\nMatchup possession statistics:")
display(
    diversity["MATCHUP_POSSESSIONS"].describe()
)

print("\nDiversity distribution:")
display(
    diversity["POSITIONAL_MATCHUP_DIVERSITY"].describe()
)

print("\nHighest matchup diversity:")
display(
    diversity.nlargest(
        10, "POSITIONAL_MATCHUP_DIVERSITY"
    )
)

print("\nLowest valid matchup diversity:")
display(
    diversity[
        diversity["VALID_FOR_MODEL"]
    ].nsmallest(
        10, "POSITIONAL_MATCHUP_DIVERSITY"
    )
)


Total defenders: 582
Valid defenders: 503

Coverage statistics:


count    5.820000e+02
mean     1.000000e+00
std      1.027867e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: ROLLUP_COVERAGE, dtype: float64


Matchup possession statistics:


count     582.000000
mean     2006.947766
std      1530.139541
min         4.000000
25%       571.325000
50%      1865.350000
75%      3170.725000
max      5668.400000
Name: MATCHUP_POSSESSIONS, dtype: float64


Diversity distribution:


count    503.000000
mean      82.598431
std       10.803001
min       56.658883
25%       74.653312
50%       82.056754
75%       92.344844
max       99.832465
Name: POSITIONAL_MATCHUP_DIVERSITY, dtype: float64


Highest matchup diversity:


POSITION,DEF_PLAYER_ID,MATCHUP_SHARE_G,MATCHUP_SHARE_F,MATCHUP_SHARE_C,MATCHUP_POSSESSIONS,MATCHUP_DIVERSITY_RAW,ROLLUP_TOTAL_POSS,ROLLUP_COVERAGE,VALID_FOR_MODEL,POSITIONAL_MATCHUP_DIVERSITY
330,1631131,0.305666,0.352844,0.341490,907.2,99.832465,907.2,1.0,True,99.832465
232,1630264,0.318142,0.377301,0.304557,1928.7,99.598493,1928.7,1.0,True,99.598493
414,1641765,0.329635,0.378665,0.291700,2019.2,99.482286,2019.2,1.0,True,99.482286
265,1630578,0.304329,0.384647,0.311024,4555.6,99.469757,4555.6,1.0,True,99.469757
484,1642366,0.282642,0.351435,0.365923,2222.6,99.445426,2222.6,1.0,True,99.445426
4,201143,0.288625,0.380873,0.330502,1860.2,99.418027,1860.2,1.0,True,99.418027
530,1642867,0.288682,0.387805,0.323513,2382.9,99.316746,2382.9,1.0,True,99.316746
223,1630214,0.282457,0.334072,0.383471,315.8,99.299831,315.8,1.0,True,99.299831
37,203486,0.324588,0.287065,0.388347,327.8,99.290435,327.8,1.0,True,99.290435
353,1631218,0.274984,0.360570,0.364446,954.6,99.278498,954.6,1.0,True,99.278498



Lowest valid matchup diversity:


POSITION,DEF_PLAYER_ID,MATCHUP_SHARE_G,MATCHUP_SHARE_F,MATCHUP_SHARE_C,MATCHUP_POSSESSIONS,MATCHUP_DIVERSITY_RAW,ROLLUP_TOTAL_POSS,ROLLUP_COVERAGE,VALID_FOR_MODEL,POSITIONAL_MATCHUP_DIVERSITY
552,1642918,0.798088,0.145698,0.056214,261.5,56.658883,261.5,1.0,True,56.658883
524,1642859,0.775147,0.189298,0.035555,1119.4,57.448861,1119.4,1.0,True,57.448861
63,204456,0.767718,0.196714,0.035567,1917.5,58.387725,1917.5,1.0,True,58.387725
158,1629162,0.756486,0.206022,0.037493,528.1,60.047870,528.1,1.0,True,60.047870
505,1642484,0.760156,0.198492,0.041352,411.1,60.180937,411.1,1.0,True,60.180937
464,1642278,0.754361,0.205130,0.040509,1352.8,60.755860,1352.8,1.0,True,60.755860
359,1631245,0.757701,0.198704,0.043595,1821.3,60.795431,1821.3,1.0,True,60.795431
540,1642880,0.732297,0.234361,0.033341,1235.7,62.040645,1235.7,1.0,True,62.040645
471,1642347,0.755459,0.191559,0.052982,3544.6,62.266416,3544.6,1.0,True,62.266416
245,1630540,0.751372,0.199466,0.049162,2058.5,62.301424,2058.5,1.0,True,62.301424


In [8]:

# Join the new defensive feature to our eligible players.
from nba_archetypes.config import INTERIM_DATA_DIR
players = pd.read_parquet(
    INTERIM_DATA_DIR / "eligible_players.parquet"
)
eligible_diversity = players[
    ["PLAYER_ID", "PLAYER_NAME"]
].merge(
    diversity,
    left_on="PLAYER_ID",
    right_on="DEF_PLAYER_ID",
    how="left",
    validate="one_to_one",
)

valid = eligible_diversity[
    "POSITIONAL_MATCHUP_DIVERSITY"
].notna()

print("Eligible players:", len(eligible_diversity))
print("Eligible players with matchup data:",
      eligible_diversity["MATCHUP_POSSESSIONS"].notna().sum())
print("Eligible players with valid diversity:", valid.sum())
print("Missing or insufficient data:", (~valid).sum())

# See how the exposure threshold affects eligibility.
print("\nCoverage at different possession thresholds:")

for threshold in [200, 500, 1000]:
    count = eligible_diversity[
        "MATCHUP_POSSESSIONS"
    ].ge(threshold).sum()

    print(f"{threshold} possessions: {count} players")

# Inspect recognizable NBA players at each extreme.
columns = [
    "PLAYER_NAME",
    "MATCHUP_SHARE_G",
    "MATCHUP_SHARE_F",
    "MATCHUP_SHARE_C",
    "MATCHUP_POSSESSIONS",
    "POSITIONAL_MATCHUP_DIVERSITY",
]

print("\nHighest diversity — eligible players:")
display(
    eligible_diversity.loc[valid]
    .nlargest(10, "POSITIONAL_MATCHUP_DIVERSITY")[columns]
    .round(3)
)

print("\nLowest diversity — eligible players:")
display(
    eligible_diversity.loc[valid]
    .nsmallest(10, "POSITIONAL_MATCHUP_DIVERSITY")[columns]
    .round(3)
)


Eligible players: 450
Eligible players with matchup data: 450
Eligible players with valid diversity: 450
Missing or insufficient data: 0

Coverage at different possession thresholds:
200 possessions: 450 players
500 possessions: 446 players
1000 possessions: 378 players

Highest diversity — eligible players:


,PLAYER_NAME,MATCHUP_SHARE_G,MATCHUP_SHARE_F,MATCHUP_SHARE_C,MATCHUP_POSSESSIONS,POSITIONAL_MATCHUP_DIVERSITY
349,Oscar Tshiebwe,0.306,0.353,0.341,907.2,99.832
23,Anthony Gill,0.318,0.377,0.305,1928.7,99.598
347,Olivier-Maxence Prosper,0.330,0.379,0.292,2019.2,99.482
12,Alperen Sengun,0.304,0.385,0.311,4555.6,99.470
367,Quinten Post,0.283,0.351,0.366,2222.6,99.445
9,Al Horford,0.289,0.381,0.331,1860.2,99.418
78,Collin Murray-Boyles,0.289,0.388,0.324,2382.9,99.317
416,Trayce Jackson-Davis,0.275,0.361,0.364,954.6,99.278
387,Sandro Mamukelashvili,0.296,0.393,0.311,3399.6,99.268
373,Robert Williams III,0.278,0.385,0.337,1962.4,99.198



Lowest diversity — eligible players:


,PLAYER_NAME,MATCHUP_SHARE_G,MATCHUP_SHARE_F,MATCHUP_SHARE_C,MATCHUP_POSSESSIONS,POSITIONAL_MATCHUP_DIVERSITY
192,Jase Richardson,0.775,0.189,0.036,1119.4,57.449
402,T.J. McConnell,0.768,0.197,0.036,1917.5,58.388
224,Jordan McLaughlin,0.756,0.206,0.037,528.1,60.048
427,Tyler Kolek,0.754,0.205,0.041,1352.8,60.756
366,Quenton Jackson,0.758,0.199,0.044,1821.3,60.795
244,Kam Jones,0.732,0.234,0.033,1235.7,62.041
182,Jamal Shead,0.755,0.192,0.053,3544.6,62.266
317,Miles McBride,0.751,0.199,0.049,2058.5,62.301
430,Tyrese Proctor,0.714,0.260,0.026,1046.9,62.335
209,Jevon Carter,0.741,0.215,0.043,1699.9,62.713


In [10]:
# Keep the three position categories.
time_data = rollup[
    rollup["POSITION"].isin(["G", "F", "C"])
].copy()

time_data["PERCENT_OF_TIME"] = pd.to_numeric(
    time_data["PERCENT_OF_TIME"]
)

# One row per defender, one column per position.
time_shares = time_data.pivot(
    index="DEF_PLAYER_ID",
    columns="POSITION",
    values="PERCENT_OF_TIME",
)

time_shares = (
    time_shares.reindex(columns=["G", "F", "C"])
    .fillna(0)
)

# Check the source percentages.
total_share = time_shares.sum(axis=1)

print(
    "Largest deviation from 1:",
    (total_share - 1).abs().max()
)

if not np.allclose(total_share, 1, atol=0.001):
    raise ValueError("Matchup time shares do not sum to 1.")

# Normalize minor rounding differences.
time_shares = time_shares.div(
    total_share, axis=0
)

# Shannon entropy, normalized to 0–100.
p = time_shares.to_numpy()

safe_p = np.where(p > 0, p, 1.0)

time_entropy = (
    -np.sum(p * np.log(safe_p), axis=1)
    / np.log(3)
    * 100
)

time_diversity = pd.DataFrame({
    "DEF_PLAYER_ID": time_shares.index,
    "MATCHUP_DIVERSITY_TIME": time_entropy,
}).reset_index(drop=True)

display(time_diversity.head())

comparison = (
    players[["PLAYER_ID", "PLAYER_NAME"]]
    .merge(
        diversity[
            [
                "DEF_PLAYER_ID",
                "POSITIONAL_MATCHUP_DIVERSITY",
            ]
        ],
        left_on="PLAYER_ID",
        right_on="DEF_PLAYER_ID",
        validate="one_to_one",
    )
    .merge(
        time_diversity,
        on="DEF_PLAYER_ID",
        validate="one_to_one",
    )
)

assert len(comparison) == 450

pos_col = "POSITIONAL_MATCHUP_DIVERSITY"
time_col = "MATCHUP_DIVERSITY_TIME"

# Compare the numerical scores.
comparison["ABS_DIFFERENCE"] = (
    comparison[pos_col] - comparison[time_col]
).abs()

# Compare player rankings (highest diversity first).
comparison["POSSESSION_RANK"] = (
    comparison[pos_col].rank(
        ascending=False, method="min"
    )
)

comparison["TIME_RANK"] = (
    comparison[time_col].rank(
        ascending=False, method="min"
    )
)

comparison["RANK_CHANGE"] = (
    comparison["POSSESSION_RANK"]
    - comparison["TIME_RANK"]
).abs()

print("Pearson correlation:",
      round(comparison[pos_col].corr(comparison[time_col]), 4))

print("Spearman rank correlation:",
      round(
          comparison[pos_col].corr(
              comparison[time_col], method="spearman"
          ), 4
      ))

print("Mean absolute score difference:",
      round(comparison["ABS_DIFFERENCE"].mean(), 3))

print("Median rank change:",
      comparison["RANK_CHANGE"].median())

print("\nPlayers with largest ranking changes:")

display(
    comparison.nlargest(15, "RANK_CHANGE")[
        [
            "PLAYER_NAME",
            pos_col,
            time_col,
            "POSSESSION_RANK",
            "TIME_RANK",
            "RANK_CHANGE",
        ]
    ].round(2)
)

Largest deviation from 1: 1.0014211682118912e-13


,DEF_PLAYER_ID,MATCHUP_DIVERSITY_TIME
0,2544,86.781880
1,101108,69.874924
2,200768,79.552685
3,201142,83.437061
4,201143,98.173528


Pearson correlation: 0.9913
Spearman rank correlation: 0.9929
Mean absolute score difference: 2.057
Median rank change: 9.0

Players with largest ranking changes:


,PLAYER_NAME,POSITIONAL_MATCHUP_DIVERSITY,MATCHUP_DIVERSITY_TIME,POSSESSION_RANK,TIME_RANK,RANK_CHANGE
445,Zach Edey,89.68,81.46,131.0,215.0,84.0
68,Chaney Johnson,95.09,96.79,93.0,37.0,56.0
290,Lindy Waters III,74.36,74.77,345.0,302.0,43.0
71,Christian Braun,77.80,73.49,283.0,325.0,42.0
38,Bobby Portis Jr.,96.61,97.44,68.0,28.0,40.0
337,Nikola Jović,95.02,95.68,95.0,55.0,40.0
415,Trae Young,73.44,65.36,356.0,395.0,39.0
129,Egor Dëmin,74.55,74.41,342.0,304.0,38.0
448,Ziaire Williams,76.06,76.42,315.0,277.0,38.0
72,Christian Koloko,96.75,92.36,63.0,100.0,37.0


In [14]:
from nba_archetypes.config import PROCESSED_DATA_DIR
features = pd.read_parquet(
    PROCESSED_DATA_DIR / "player_style_features.parquet"
)

defense_cols = [
    "DEFENSIVE_DISRUPTION_SCORE",
    "PERIMETER_DEFENSE_SCORE",
    "INTERIOR_DEFENSE_SCORE",
    "DEFENSIVE_REBOUNDING_SCORE",
]

defense_corr = features[defense_cols].corr()

display(defense_corr.round(3))

print(
    "Interior Defense vs Defensive Rebounding:",
    round(
        defense_corr.loc[
            "INTERIOR_DEFENSE_SCORE",
            "DEFENSIVE_REBOUNDING_SCORE",
        ],
        3,
    )
)


defense_experiment = features[
    ["PLAYER_ID", "PLAYER_NAME"] + defense_cols
].copy()

# Combine the two previously standardized dimensions.
interior_combined = (
    defense_experiment["INTERIOR_DEFENSE_SCORE"]
    + defense_experiment["DEFENSIVE_REBOUNDING_SCORE"]
) / 2

# Standardize the combined dimension.
defense_experiment["INTERIOR_PRESENCE_SCORE"] = (
    interior_combined - interior_combined.mean()
) / interior_combined.std(ddof=0)

display(
    defense_experiment[
        [
            "INTERIOR_DEFENSE_SCORE",
            "DEFENSIVE_REBOUNDING_SCORE",
            "INTERIOR_PRESENCE_SCORE",
        ]
    ].corr().round(3)
)


,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_DEFENSE_SCORE,DEFENSIVE_REBOUNDING_SCORE
DEFENSIVE_DISRUPTION_SCORE,1.000,0.235,-0.078,-0.111
PERIMETER_DEFENSE_SCORE,0.235,1.000,-0.765,-0.538
INTERIOR_DEFENSE_SCORE,-0.078,-0.765,1.000,0.668
DEFENSIVE_REBOUNDING_SCORE,-0.111,-0.538,0.668,1.000


Interior Defense vs Defensive Rebounding: 0.668


,INTERIOR_DEFENSE_SCORE,DEFENSIVE_REBOUNDING_SCORE,INTERIOR_PRESENCE_SCORE
INTERIOR_DEFENSE_SCORE,1.000,0.668,0.913
DEFENSIVE_REBOUNDING_SCORE,0.668,1.000,0.913
INTERIOR_PRESENCE_SCORE,0.913,0.913,1.000


In [15]:

# Start with the eligible-player defensive scores.
defense_experiment = features[
    [
        "PLAYER_ID",
        "PLAYER_NAME",
        "DEFENSIVE_DISRUPTION_SCORE",
        "PERIMETER_DEFENSE_SCORE",
        "INTERIOR_DEFENSE_SCORE",
        "DEFENSIVE_REBOUNDING_SCORE",
    ]
].copy()

# Merge time-weighted matchup diversity.
defense_experiment = defense_experiment.merge(
    time_diversity,
    left_on="PLAYER_ID",
    right_on="DEF_PLAYER_ID",
    how="left",
    validate="one_to_one",
)

assert len(defense_experiment) == 450
assert defense_experiment["MATCHUP_DIVERSITY_TIME"].notna().all()

# Combine Interior Defense and Defensive Rebounding.
interior = defense_experiment[
    ["INTERIOR_DEFENSE_SCORE", "DEFENSIVE_REBOUNDING_SCORE"]
].mean(axis=1)

defense_experiment["INTERIOR_PRESENCE_SCORE"] = (
    (interior - interior.mean())
    / interior.std(ddof=0)
)

# Standardize matchup diversity using eligible players.
matchup = defense_experiment["MATCHUP_DIVERSITY_TIME"]

defense_experiment["MATCHUP_DIVERSITY_SCORE"] = (
    (matchup - matchup.mean())
    / matchup.std(ddof=0)
)

# Candidate four-dimensional representation.
defense_style_cols = [
    "DEFENSIVE_DISRUPTION_SCORE",
    "PERIMETER_DEFENSE_SCORE",
    "INTERIOR_PRESENCE_SCORE",
    "MATCHUP_DIVERSITY_SCORE",
]

print("Correlation matrix:")
display(
    defense_experiment[defense_style_cols]
    .corr()
    .round(3)
)

print("\nDimension distributions:")
display(
    defense_experiment[defense_style_cols]
    .describe()
    .round(3)
)


Correlation matrix:


,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_PRESENCE_SCORE,MATCHUP_DIVERSITY_SCORE
DEFENSIVE_DISRUPTION_SCORE,1.000,0.235,-0.103,-0.131
PERIMETER_DEFENSE_SCORE,0.235,1.000,-0.714,-0.566
INTERIOR_PRESENCE_SCORE,-0.103,-0.714,1.000,0.777
MATCHUP_DIVERSITY_SCORE,-0.131,-0.566,0.777,1.000



Dimension distributions:


,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_PRESENCE_SCORE,MATCHUP_DIVERSITY_SCORE
count,450.000,450.000,450.000,450.000
mean,-0.000,0.000,-0.000,0.000
std,1.001,1.001,1.001,1.001
min,-2.792,-2.358,-2.091,-2.475
25%,-0.640,-0.709,-0.775,-0.704
50%,-0.041,0.001,-0.236,0.010
75%,0.636,0.622,0.552,0.897
max,3.649,2.652,3.090,1.624


In [16]:

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
)

# Model A: Three core defensive dimensions.
core_cols = [
    "DEFENSIVE_DISRUPTION_SCORE",
    "PERIMETER_DEFENSE_SCORE",
    "INTERIOR_PRESENCE_SCORE",
]

# Model B: Add positional matchup diversity.
extended_cols = core_cols + [
    "MATCHUP_DIVERSITY_SCORE"
]

X_core = defense_experiment[core_cols].to_numpy()
X_extended = defense_experiment[extended_cols].to_numpy()

defense_models = {}
results = []

for k in range(3, 7):

    core = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20,
    )

    extended = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20,
    )

    core_labels = core.fit_predict(X_core)
    extended_labels = extended.fit_predict(X_extended)

    defense_models[("Core", k)] = core
    defense_models[("Extended", k)] = extended

    core_sizes = pd.Series(core_labels).value_counts()
    extended_sizes = pd.Series(extended_labels).value_counts()

    results.append({
        "k": k,
        "Core Silhouette": silhouette_score(
            X_core, core_labels
        ),
        "Extended Silhouette": silhouette_score(
            X_extended, extended_labels
        ),
        "ARI": adjusted_rand_score(
            core_labels, extended_labels
        ),
        "Core Min": core_sizes.min(),
        "Core Max": core_sizes.max(),
        "Extended Min": extended_sizes.min(),
        "Extended Max": extended_sizes.max(),
    })

defense_results = pd.DataFrame(results)

display(defense_results.round(3))


,k,Core Silhouette,Extended Silhouette,ARI,Core Min,Core Max,Extended Min,Extended Max
0,3,0.292,0.269,0.700,116,184,138,170
1,4,0.286,0.253,0.592,96,125,100,129
2,5,0.281,0.231,0.454,63,123,74,117
3,6,0.278,0.240,0.504,53,121,50,117


In [17]:

# Add time-weighted G/F/C matchup shares.
role_experiment = defense_experiment.merge(
    time_shares.add_prefix("TIME_SHARE_"),
    left_on="PLAYER_ID",
    right_index=True,
    validate="one_to_one",
)

# Two contrasts preserve the information in all three shares.
role_experiment["GUARD_CENTER_BALANCE"] = (
    role_experiment["TIME_SHARE_G"]
    - role_experiment["TIME_SHARE_C"]
)

role_experiment["FORWARD_FOCUS"] = (
    role_experiment["TIME_SHARE_F"]
    - (
        role_experiment["TIME_SHARE_G"]
        + role_experiment["TIME_SHARE_C"]
    ) / 2
)

assignment_cols = [
    "GUARD_CENTER_BALANCE",
    "FORWARD_FOCUS",
]

# Standardize the two assignment dimensions.
assignment = role_experiment[assignment_cols]

assignment_scaled = (
    (assignment - assignment.mean())
    / assignment.std(ddof=0)
)

# Balance the two assignment dimensions as a group.
assignment_scaled = assignment_scaled / np.sqrt(2)

# Combine with the three core dimensions.
X_role = np.column_stack([
    role_experiment[core_cols].to_numpy(),
    assignment_scaled.to_numpy(),
])

print("Shape:", X_role.shape)

display(
    role_experiment[
        core_cols + assignment_cols
    ].corr().round(3)
)

k = 4

role_model = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=20,
)

role_labels = role_model.fit_predict(X_role)

core_labels = defense_models[("Core", k)].labels_

print(
    "Core vs. assignment-aware ARI:",
    round(adjusted_rand_score(core_labels, role_labels), 3)
)

print(
    "Assignment-aware silhouette:",
    round(silhouette_score(X_role, role_labels), 3)
)

role_experiment["ROLE_CLUSTER"] = role_labels

print("\nCluster sizes:")
display(
    role_experiment["ROLE_CLUSTER"]
    .value_counts()
    .sort_index()
)

print("\nDefensive profiles:")
display(
    role_experiment.groupby("ROLE_CLUSTER")[
        core_cols + [
            "TIME_SHARE_G",
            "TIME_SHARE_F",
            "TIME_SHARE_C",
        ]
    ].mean().round(3)
)


Shape: (450, 5)


,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_PRESENCE_SCORE,GUARD_CENTER_BALANCE,FORWARD_FOCUS
DEFENSIVE_DISRUPTION_SCORE,1.000,0.235,-0.103,0.181,-0.173
PERIMETER_DEFENSE_SCORE,0.235,1.000,-0.714,0.657,-0.221
INTERIOR_PRESENCE_SCORE,-0.103,-0.714,1.000,-0.880,0.357
GUARD_CENTER_BALANCE,0.181,0.657,-0.880,1.000,-0.414
FORWARD_FOCUS,-0.173,-0.221,0.357,-0.414,1.000


Core vs. assignment-aware ARI: 0.311
Assignment-aware silhouette: 0.253

Cluster sizes:


ROLE_CLUSTER
0     95
1    149
2     86
3    120
Name: count, dtype: int64


Defensive profiles:


,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_PRESENCE_SCORE,TIME_SHARE_G,TIME_SHARE_F,TIME_SHARE_C
ROLE_CLUSTER,,,,,,
0,1.252,0.700,-0.506,0.617,0.313,0.069
1,-0.309,-0.258,0.088,0.450,0.436,0.114
2,-0.273,-1.249,1.626,0.208,0.361,0.431
3,-0.412,0.662,-0.874,0.661,0.290,0.049


In [18]:

# Existing k=4 core KMeans model.
core_model = defense_models[("Core", 4)]

core_view = role_experiment[
    [
        "PLAYER_NAME",
        *core_cols,
        "TIME_SHARE_G",
        "TIME_SHARE_F",
        "TIME_SHARE_C",
    ]
].copy()

core_view["CLUSTER"] = core_model.labels_

# Cluster profiles, including assignment information
# that was NOT used to fit the core model.
profiles = core_view.groupby("CLUSTER")[
    core_cols + [
        "TIME_SHARE_G",
        "TIME_SHARE_F",
        "TIME_SHARE_C",
    ]
].mean()

print("Core defensive cluster profiles:")
display(profiles.round(3))

# Representative players nearest each KMeans center.
for cluster in range(4):

    group = core_view[
        core_view["CLUSTER"] == cluster
    ]

    center = core_model.cluster_centers_[cluster]

    distances = np.linalg.norm(
        group[core_cols].to_numpy() - center,
        axis=1,
    )

    nearest = group.iloc[
        np.argsort(distances)[:7]
    ]

    print(f"\nCore Defensive Cluster {cluster}")
    display(
        nearest[
            ["PLAYER_NAME", *core_cols]
        ].round(2)
    )


Core defensive cluster profiles:


,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_PRESENCE_SCORE,TIME_SHARE_G,TIME_SHARE_F,TIME_SHARE_C
CLUSTER,,,,,,
0,-0.326,-1.233,1.522,0.238,0.378,0.384
1,0.664,-0.165,0.015,0.519,0.372,0.109
2,-0.921,0.035,-0.485,0.556,0.364,0.079
3,0.581,1.270,-0.833,0.630,0.312,0.058



Core Defensive Cluster 0


,PLAYER_NAME,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_PRESENCE_SCORE
275,Kristaps Porziņģis,-0.49,-1.47,1.60
7,Adem Bona,-0.08,-1.02,1.52
282,Lachlan Olbrich,0.04,-1.16,1.51
245,Karl-Anthony Towns,-0.04,-1.01,1.41
325,Myles Turner,-0.11,-1.57,1.66
246,Karlo Matković,0.02,-0.96,1.44
111,Domantas Sabonis,-0.72,-1.08,1.69



Core Defensive Cluster 1


,PLAYER_NAME,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_PRESENCE_SCORE
425,Tyler Burton,0.64,-0.27,0.05
435,Vince Williams Jr.,0.66,0.03,0.03
369,Rasheer Fleming,0.88,-0.15,-0.14
100,Dejounte Murray,0.59,0.05,-0.16
406,Taylor Hendricks,0.60,-0.05,0.33
104,Derrick Jones Jr.,0.46,-0.40,-0.12
344,OG Anunoby,0.57,0.19,-0.04



Core Defensive Cluster 2


,PLAYER_NAME,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_PRESENCE_SCORE
33,Bennedict Mathurin,-1.17,0.03,-0.53
298,Malaki Branham,-1.13,0.15,-0.58
28,Ayo Dosunmu,-1.10,-0.06,-0.69
216,John Poulakidas,-1.21,-0.15,-0.49
319,Mohamed Diawara,-1.01,-0.30,-0.52
203,Jerami Grant,-1.12,0.21,-0.78
252,Keldon Johnson,-0.91,-0.05,-0.08



Core Defensive Cluster 3


,PLAYER_NAME,DEFENSIVE_DISRUPTION_SCORE,PERIMETER_DEFENSE_SCORE,INTERIOR_PRESENCE_SCORE
114,Donovan Mitchell,0.57,1.26,-0.86
77,Collin Gillespie,0.75,1.29,-0.86
94,De'Aaron Fox,0.70,1.22,-0.95
249,Keaton Wallace,0.41,1.39,-0.85
210,Jimmy Butler III,0.37,1.07,-0.86
158,Ja Morant,0.43,1.04,-0.98
343,Norman Powell,0.55,1.04,-1.05


In [19]:
# Get the total recorded matchup time for each defender.
totals = rollup.loc[
    rollup["POSITION"] == "TOTAL",
    ["DEF_PLAYER_ID", "MATCHUP_MIN"],
].copy()

parts = totals["MATCHUP_MIN"].str.split(
    ":", expand=True
).astype(float)

totals["SECONDS"] = parts[0] * 60 + parts[1]

total_seconds = (
    totals.set_index("DEF_PLAYER_ID")["SECONDS"]
    .reindex(time_shares.index)
)

assert total_seconds.notna().all()

# League-wide opportunity baseline, weighted by recorded time.
empirical_baseline = (
    time_shares.mul(total_seconds, axis=0).sum()
    / total_seconds.sum()
)

theoretical_baseline = pd.Series({
    "G": 0.40,
    "F": 0.40,
    "C": 0.20,
})

print("Theoretical baseline:")
display(theoretical_baseline)

print("Actual league-wide baseline:")
display(empirical_baseline.round(3))


def entropy_score(shares):
    p = shares.to_numpy(dtype=float)
    safe_p = np.where(p > 0, p, 1.0)

    scores = -np.sum(
        p * np.log(safe_p), axis=1
    ) / np.log(3) * 100

    return pd.Series(scores, index=shares.index)


def relative_balance_score(shares, baseline):
    # Adjust each share relative to its expected exposure.
    adjusted = shares.div(baseline, axis=1)

    # Renormalize to proportions.
    adjusted = adjusted.div(
        adjusted.sum(axis=1), axis=0
    )

    return entropy_score(adjusted)


scores = pd.DataFrame({
    "Original": entropy_score(time_shares),
    "Theoretical_Adjusted": relative_balance_score(
        time_shares, theoretical_baseline
    ),
    "Empirical_Adjusted": relative_balance_score(
        time_shares, empirical_baseline
    ),
})

eligible_scores = players[
    ["PLAYER_ID", "PLAYER_NAME"]
].merge(
    scores.rename_axis("DEF_PLAYER_ID").reset_index(),
    left_on="PLAYER_ID",
    right_on="DEF_PLAYER_ID",
    validate="one_to_one",
)

print("Spearman rank correlations:")
display(
    eligible_scores[
        ["Original", "Theoretical_Adjusted",
         "Empirical_Adjusted"]
    ].corr(method="spearman").round(3)
)

print("Highest empirical-adjusted scores:")
display(
    eligible_scores.nlargest(
        10, "Empirical_Adjusted"
    )[
        ["PLAYER_NAME", "Original",
         "Theoretical_Adjusted",
         "Empirical_Adjusted"]
    ].round(2)
)


Theoretical baseline:


G    0.4
F    0.4
C    0.2
dtype: float64

Actual league-wide baseline:


POSITION
G    0.497
F    0.360
C    0.143
dtype: float64

Spearman rank correlations:


,Original,Theoretical_Adjusted,Empirical_Adjusted
Original,1.000,0.722,0.144
Theoretical_Adjusted,0.722,1.000,0.751
Empirical_Adjusted,0.144,0.751,1.000


Highest empirical-adjusted scores:


,PLAYER_NAME,Original,Theoretical_Adjusted,Empirical_Adjusted
190,Jarred Vanderbilt,88.09,95.87,99.75
215,John Konchar,88.09,96.12,99.62
65,Carter Bryant,87.77,95.87,99.52
362,Pete Nance,91.22,98.38,99.52
10,Alex Caruso,87.28,95.48,99.39
232,Josh Minott,89.39,97.19,99.33
116,Dorian Finney-Smith,89.98,97.61,99.32
397,Spencer Jones,91.37,98.48,99.32
165,Jae'Sean Tate,89.74,97.44,99.32
160,Jabari Smith Jr.,91.58,98.60,99.24


In [20]:
role_profiles = (
    players[["PLAYER_ID", "PLAYER_NAME"]]
    .merge(
        time_shares.reset_index(),
        left_on="PLAYER_ID",
        right_on="DEF_PLAYER_ID",
        validate="one_to_one",
    )
)

for position in ["G", "F", "C"]:
    role_profiles[f"{position}_VS_LEAGUE_PP"] = (
        role_profiles[position]
        - empirical_baseline[position]
    ) * 100

names = [
    "Jarred Vanderbilt",
    "Alex Caruso",
    "Alperen Sengun",
    "T.J. McConnell",
    "Myles Turner",
]

display(
    role_profiles[
        role_profiles["PLAYER_NAME"].isin(names)
    ][
        [
            "PLAYER_NAME",
            "G", "F", "C",
            "G_VS_LEAGUE_PP",
            "F_VS_LEAGUE_PP",
            "C_VS_LEAGUE_PP",
        ]
    ].round(3)
)


,PLAYER_NAME,G,F,C,G_VS_LEAGUE_PP,F_VS_LEAGUE_PP,C_VS_LEAGUE_PP
10,Alex Caruso,0.512,0.374,0.115,1.442,1.387,-2.829
12,Alperen Sengun,0.280,0.383,0.337,-21.711,2.333,19.378
190,Jarred Vanderbilt,0.536,0.331,0.132,3.912,-2.862,-1.050
325,Myles Turner,0.222,0.395,0.384,-27.569,3.494,24.075
402,T.J. McConnell,0.806,0.168,0.026,30.828,-19.144,-11.685


In [26]:
from nba_api.stats.endpoints import leaguedashptdefend
from time import sleep

season = "2025-26"

categories = [
    "Overall",
    "Less Than 6Ft",
    "3 Pointers",
]

defended_tables = {}

for category in categories:

    df = leaguedashptdefend.LeagueDashPtDefend(
        season=season,
        season_type_all_star="Regular Season",
        defense_category=category,
        per_mode_simple="Totals",
        timeout=60,
    ).get_data_frames()[0]

    defended_tables[category] = df

    print(f"\n{category}: {len(df)} rows")
    print(df.columns.tolist())
    sleep(1.5)

    display(
        df.head()
    )


Overall: 580 rows
['CLOSE_DEF_PERSON_ID', 'PLAYER_NAME', 'PLAYER_LAST_TEAM_ID', 'PLAYER_LAST_TEAM_ABBREVIATION', 'PLAYER_POSITION', 'AGE', 'GP', 'G', 'FREQ', 'D_FGM', 'D_FGA', 'D_FG_PCT', 'NORMAL_FG_PCT', 'PCT_PLUSMINUS']


,CLOSE_DEF_PERSON_ID,PLAYER_NAME,PLAYER_LAST_TEAM_ID,PLAYER_LAST_TEAM_ABBREVIATION,PLAYER_POSITION,AGE,GP,G,FREQ,D_FGM,D_FGA,D_FG_PCT,NORMAL_FG_PCT,PCT_PLUSMINUS
0,1642270,Donovan Clingan,1610612757,POR,C,22.0,77,77,1.0,656,1453,0.451,0.497,-0.046
1,203497,Rudy Gobert,1610612750,MIN,C,34.0,76,76,1.0,538,1223,0.440,0.489,-0.050
2,1630578,Alperen Sengun,1610612745,HOU,C,23.0,71,71,1.0,617,1219,0.506,0.484,0.023
3,1642852,Derik Queen,1610612740,NOP,C,21.0,81,81,1.0,561,1156,0.485,0.489,-0.003
4,1641705,Victor Wembanyama,1610612759,SAS,F-C,22.0,64,64,1.0,417,1092,0.382,0.492,-0.110



Less Than 6Ft: 576 rows
['CLOSE_DEF_PERSON_ID', 'PLAYER_NAME', 'PLAYER_LAST_TEAM_ID', 'PLAYER_LAST_TEAM_ABBREVIATION', 'PLAYER_POSITION', 'AGE', 'GP', 'G', 'FREQ', 'FGM_LT_06', 'FGA_LT_06', 'LT_06_PCT', 'NS_LT_06_PCT', 'PLUSMINUS']


,CLOSE_DEF_PERSON_ID,PLAYER_NAME,PLAYER_LAST_TEAM_ID,PLAYER_LAST_TEAM_ABBREVIATION,PLAYER_POSITION,AGE,GP,G,FREQ,FGM_LT_06,FGA_LT_06,LT_06_PCT,NS_LT_06_PCT,PLUSMINUS
0,1642270,Donovan Clingan,1610612757,POR,C,22.0,77,77,0.383,291,556,0.523,0.643,-0.120
1,1630578,Alperen Sengun,1610612745,HOU,C,23.0,71,71,0.427,326,521,0.626,0.640,-0.015
2,1642852,Derik Queen,1610612740,NOP,C,21.0,81,80,0.432,305,499,0.611,0.647,-0.036
3,203497,Rudy Gobert,1610612750,MIN,C,34.0,76,76,0.407,261,498,0.524,0.635,-0.111
4,1630643,Jay Huff,1610612754,IND,C,28.0,81,80,0.510,250,496,0.504,0.634,-0.130



3 Pointers: 575 rows
['CLOSE_DEF_PERSON_ID', 'PLAYER_NAME', 'PLAYER_LAST_TEAM_ID', 'PLAYER_LAST_TEAM_ABBREVIATION', 'PLAYER_POSITION', 'AGE', 'GP', 'G', 'FREQ', 'FG3M', 'FG3A', 'FG3_PCT', 'NS_FG3_PCT', 'PLUSMINUS']


,CLOSE_DEF_PERSON_ID,PLAYER_NAME,PLAYER_LAST_TEAM_ID,PLAYER_LAST_TEAM_ABBREVIATION,PLAYER_POSITION,AGE,GP,G,FREQ,FG3M,FG3A,FG3_PCT,NS_FG3_PCT,PLUSMINUS
0,1642270,Donovan Clingan,1610612757,POR,C,22.0,77,77,0.343,196,499,0.393,0.356,0.037
1,1627750,Jamal Murray,1610612743,DEN,G,29.0,75,75,0.509,163,474,0.344,0.362,-0.018
2,1628401,Derrick White,1610612738,BOS,G,31.0,77,75,0.451,160,471,0.340,0.359,-0.019
3,1628973,Jalen Brunson,1610612752,NYK,G,29.0,73,73,0.566,170,454,0.374,0.363,0.012
4,1631157,Ryan Rollins,1610612749,MIL,G,23.0,74,74,0.491,146,418,0.349,0.360,-0.011


In [28]:
rebounding = pd.read_parquet(RAW_DATA_DIR / "rebounding_stats.parquet")

print(rebounding.columns.tolist())

reb_cols = [
    col for col in rebounding.columns
    if "DREB" in col.upper()
]

display(rebounding[reb_cols].head())



['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'GP', 'W', 'L', 'MIN', 'OREB', 'OREB_CONTEST', 'OREB_UNCONTEST', 'OREB_CONTEST_PCT', 'OREB_CHANCES', 'OREB_CHANCE_PCT', 'OREB_CHANCE_DEFER', 'OREB_CHANCE_PCT_ADJ', 'AVG_OREB_DIST', 'DREB', 'DREB_CONTEST', 'DREB_UNCONTEST', 'DREB_CONTEST_PCT', 'DREB_CHANCES', 'DREB_CHANCE_PCT', 'DREB_CHANCE_DEFER', 'DREB_CHANCE_PCT_ADJ', 'AVG_DREB_DIST', 'REB', 'REB_CONTEST', 'REB_UNCONTEST', 'REB_CONTEST_PCT', 'REB_CHANCES', 'REB_CHANCE_PCT', 'REB_CHANCE_DEFER', 'REB_CHANCE_PCT_ADJ', 'AVG_REB_DIST']


,DREB,DREB_CONTEST,DREB_UNCONTEST,DREB_CONTEST_PCT,DREB_CHANCES,DREB_CHANCE_PCT,DREB_CHANCE_DEFER,DREB_CHANCE_PCT_ADJ,AVG_DREB_DIST
0,34,7,27,0.206,38,0.895,5,1.030,6.7
1,184,35,149,0.190,286,0.643,53,0.790,9.1
2,41,9,32,0.220,58,0.707,10,0.854,7.3
3,158,49,109,0.310,197,0.802,20,0.893,5.7
4,45,12,33,0.267,65,0.692,11,0.833,10.0


In [29]:
category_columns = {
    "Overall": {
        "D_FGA": "OVERALL_DFGA",
        "D_FG_PCT": "OVERALL_DFG_PCT",
        "NORMAL_FG_PCT": "OVERALL_NORMAL_FG_PCT",
        "PCT_PLUSMINUS": "OVERALL_FG_DIFF",
    },
    "Less Than 6Ft": {
        "FGA_LT_06": "RIM_DFGA",
        "LT_06_PCT": "RIM_DFG_PCT",
        "NS_LT_06_PCT": "RIM_NORMAL_FG_PCT",
        "PLUSMINUS": "RIM_FG_DIFF",
    },
    "3 Pointers": {
        "FG3A": "THREE_DFGA",
        "FG3_PCT": "THREE_DFG_PCT",
        "NS_FG3_PCT": "THREE_NORMAL_FG_PCT",
        "PLUSMINUS": "THREE_FG_DIFF",
    },
}

defensive_performance = players[
    ["PLAYER_ID", "PLAYER_NAME"]
].copy()

for category, mapping in category_columns.items():

    source = defended_tables[category]

    columns = ["CLOSE_DEF_PERSON_ID"] + list(mapping)

    cleaned = source[columns].rename(
        columns={
            "CLOSE_DEF_PERSON_ID": "PLAYER_ID",
            **mapping,
        }
    )

    defensive_performance = defensive_performance.merge(
        cleaned,
        on="PLAYER_ID",
        how="left",
        validate="one_to_one",
    )

# Add existing rebounding outcomes.
rebound_columns = [
    "PLAYER_ID",
    "DREB",
    "DREB_CHANCES",
    "DREB_CHANCE_DEFER",
    "DREB_CHANCE_PCT",
    "DREB_CHANCE_PCT_ADJ",
    "DREB_CONTEST_PCT",
]

defensive_performance = defensive_performance.merge(
    rebounding[rebound_columns],
    on="PLAYER_ID",
    how="left",
    validate="one_to_one",
)

assert len(defensive_performance) == 450

print("Shape:", defensive_performance.shape)

display(defensive_performance.head())

# Coverage and shooting-volume checks.
for category in ["OVERALL", "RIM", "THREE"]:

    attempts = defensive_performance[
        f"{category}_DFGA"
    ]

    differential = defensive_performance[
        f"{category}_FG_DIFF"
    ]

    print(f"\n{category}")

    print("Players with data:", attempts.notna().sum())
    print("Players with >= 50 attempts:", attempts.ge(50).sum())
    print("Players with >= 100 attempts:", attempts.ge(100).sum())

    print("Attempt distribution:")
    display(attempts.describe())

    print("Differential distribution:")
    display(differential.describe())

    # Verify the reported difference uses the expected sign.
    calculated = (
        defensive_performance[f"{category}_DFG_PCT"]
        - defensive_performance[f"{category}_NORMAL_FG_PCT"]
    )

    mismatch = (
        (calculated - differential).abs() > 0.002
    )

    print("Differential mismatches:", mismatch.sum())


# Defensive rebounding checks.
print("\nDEFENSIVE REBOUNDING")

print(
    "Players missing rebound data:",
    defensive_performance["DREB_CHANCES"].isna().sum()
)

print(
    "Adjusted percentages above 100%:",
    defensive_performance[
        "DREB_CHANCE_PCT_ADJ"
    ].gt(1).sum()
)

print(
    "Standard percentages above 100%:",
    defensive_performance[
        "DREB_CHANCE_PCT"
    ].gt(1).sum()
)

display(
    defensive_performance[
        [
            "DREB_CHANCES",
            "DREB_CHANCE_PCT",
            "DREB_CHANCE_PCT_ADJ",
            "DREB_CONTEST_PCT",
        ]
    ].describe().round(3)
)


Shape: (450, 20)


,PLAYER_ID,PLAYER_NAME,OVERALL_DFGA,OVERALL_DFG_PCT,OVERALL_NORMAL_FG_PCT,OVERALL_FG_DIFF,RIM_DFGA,RIM_DFG_PCT,RIM_NORMAL_FG_PCT,RIM_FG_DIFF,THREE_DFGA,THREE_DFG_PCT,THREE_NORMAL_FG_PCT,THREE_FG_DIFF,DREB,DREB_CHANCES,DREB_CHANCE_DEFER,DREB_CHANCE_PCT,DREB_CHANCE_PCT_ADJ,DREB_CONTEST_PCT
0,1631260,AJ Green,856,0.493,0.457,0.036,205,0.751,0.627,0.124,418,0.392,0.359,0.034,184,286,53,0.643,0.790,0.190
1,1642358,AJ Johnson,146,0.500,0.445,0.055,44,0.682,0.607,0.075,68,0.441,0.367,0.074,41,58,10,0.707,0.854,0.220
2,203932,Aaron Gordon,372,0.419,0.475,-0.055,128,0.688,0.644,0.044,160,0.225,0.355,-0.130,158,197,20,0.802,0.893,0.310
3,1628988,Aaron Holiday,263,0.437,0.454,-0.017,65,0.754,0.639,0.115,125,0.288,0.351,-0.063,45,65,11,0.692,0.833,0.267
4,1630174,Aaron Nesmith,535,0.469,0.474,-0.005,183,0.617,0.644,-0.027,211,0.346,0.365,-0.019,139,192,27,0.724,0.842,0.158



OVERALL
Players with data: 450
Players with >= 50 attempts: 449
Players with >= 100 attempts: 434
Attempt distribution:


count     450.000000
mean      471.882222
std       259.204285
min        46.000000
25%       259.250000
50%       443.500000
75%       647.500000
max      1453.000000
Name: OVERALL_DFGA, dtype: float64

Differential distribution:


count    450.000000
mean       0.005982
std        0.039142
min       -0.111000
25%       -0.017750
50%        0.003500
75%        0.031000
max        0.142000
Name: OVERALL_FG_DIFF, dtype: float64

... (6 outputs left out)

Differential distribution:


count    450.000000
mean       0.001813
std        0.043468
min       -0.158000
25%       -0.021750
50%       -0.001500
75%        0.025000
max        0.155000
Name: THREE_FG_DIFF, dtype: float64

Differential mismatches: 0

DEFENSIVE REBOUNDING
Players missing rebound data: 0
Adjusted percentages above 100%: 24
Standard percentages above 100%: 0


,DREB_CHANCES,DREB_CHANCE_PCT,DREB_CHANCE_PCT_ADJ,DREB_CONTEST_PCT
count,450.000,450.000,450.000,450.000
mean,234.798,0.723,0.887,0.240
std,147.717,0.075,0.077,0.087
min,26.000,0.462,0.581,0.000
25%,120.250,0.677,0.836,0.181
50%,208.000,0.727,0.894,0.230
75%,313.750,0.778,0.941,0.305
max,792.000,0.900,1.077,0.509


In [32]:

made_columns = {
    "Overall": ("D_FGM", "OVERALL_DFGM"),
    "Less Than 6Ft": ("FGM_LT_06", "RIM_DFGM"),
    "3 Pointers": ("FG3M", "THREE_DFGM"),
}

for category, (original, new) in made_columns.items():

    makes = defended_tables[category][
        ["CLOSE_DEF_PERSON_ID", original]
    ].rename(
        columns={
            "CLOSE_DEF_PERSON_ID": "PLAYER_ID",
            original: new,
        }
    )

    defensive_performance = defensive_performance.merge(
        makes,
        on="PLAYER_ID",
        how="left",
        validate="one_to_one",
    )

assert len(defensive_performance) == 450

print(
    defensive_performance[
        ["OVERALL_DFGM", "RIM_DFGM", "THREE_DFGM"]
    ].isna().sum()
)


z = 1.96
k = 100  # Experimental shrinkage strength

for category in ["OVERALL", "RIM", "THREE"]:

    n = defensive_performance[
        f"{category}_DFGA"
    ].astype(float)

    made = defensive_performance[
        f"{category}_DFGM"
    ].astype(float)

    normal = defensive_performance[
        f"{category}_NORMAL_FG_PCT"
    ]

    assert ((n > 0) & (made >= 0) & (made <= n)).all()

    # Observed shooting percentage and differential.
    p = made / n
    difference = p - normal

    # Wilson confidence interval for observed FG%.
    denominator = 1 + z**2 / n

    center = (
        p + z**2 / (2 * n)
    ) / denominator

    margin = (
        z * np.sqrt(
            p * (1 - p) / n
            + z**2 / (4 * n**2)
        )
    ) / denominator

    # Subtract shooter-normal FG% from both bounds.
    defensive_performance[
        f"{category}_DIFF_CI_LOW"
    ] = center - margin - normal

    defensive_performance[
        f"{category}_DIFF_CI_HIGH"
    ] = center + margin - normal

    # Experimental shrinkage toward zero differential.
    defensive_performance[
        f"{category}_DIFF_SHRUNK"
    ] = n / (n + k) * difference

    print(f"\n{category}")

    print(
        "Intervals entirely below zero:",
        (
            defensive_performance[
                f"{category}_DIFF_CI_HIGH"
            ] < 0
        ).sum()
    )

    print(
        "Intervals entirely above zero:",
        (
            defensive_performance[
                f"{category}_DIFF_CI_LOW"
            ] > 0
        ).sum()
    )


rim_comparison = defensive_performance[
    [
        "PLAYER_NAME",
        "RIM_DFGA",
        "RIM_FG_DIFF",
        "RIM_DIFF_SHRUNK",
        "RIM_DIFF_CI_LOW",
        "RIM_DIFF_CI_HIGH",
    ]
].copy()

print("Players with most favorable raw rim differentials:")

display(
    rim_comparison.nsmallest(
        15, "RIM_FG_DIFF"
    ).round(3)
)

print("\nPlayers with most favorable shrunk differentials:")

display(
    rim_comparison.nsmallest(
        15, "RIM_DIFF_SHRUNK"
    ).round(3)
)



OVERALL_DFGM    0
RIM_DFGM        0
THREE_DFGM      0
dtype: int64

OVERALL
Intervals entirely below zero: 41
Intervals entirely above zero: 41

RIM
Intervals entirely below zero: 65
Intervals entirely above zero: 86

THREE
Intervals entirely below zero: 13
Intervals entirely above zero: 17
Players with most favorable raw rim differentials:


,PLAYER_NAME,RIM_DFGA,RIM_FG_DIFF,RIM_DIFF_SHRUNK,RIM_DIFF_CI_LOW,RIM_DIFF_CI_HIGH
156,Isaiah Stewart,255,-0.228,-0.164,-0.286,-0.167
266,Khaman Maluach,116,-0.215,-0.116,-0.302,-0.125
70,Chet Holmgren,433,-0.183,-0.149,-0.230,-0.136
73,Clint Capela,201,-0.182,-0.121,-0.249,-0.112
445,Zach Edey,45,-0.178,-0.055,-0.316,-0.036
434,Victor Wembanyama,411,-0.172,-0.138,-0.220,-0.124
219,Jonathan Isaac,86,-0.166,-0.077,-0.268,-0.061
373,Robert Williams III,219,-0.161,-0.111,-0.226,-0.095
157,Ivica Zubac,260,-0.155,-0.112,-0.215,-0.094
24,Ariel Hukporti,115,-0.147,-0.078,-0.236,-0.056



Players with most favorable shrunk differentials:


,PLAYER_NAME,RIM_DFGA,RIM_FG_DIFF,RIM_DIFF_SHRUNK,RIM_DIFF_CI_LOW,RIM_DIFF_CI_HIGH
156,Isaiah Stewart,255,-0.228,-0.164,-0.286,-0.167
70,Chet Holmgren,433,-0.183,-0.149,-0.230,-0.136
434,Victor Wembanyama,411,-0.172,-0.138,-0.220,-0.124
73,Clint Capela,201,-0.182,-0.121,-0.249,-0.112
381,Ryan Kalkbrenner,372,-0.147,-0.116,-0.198,-0.096
266,Khaman Maluach,116,-0.215,-0.116,-0.302,-0.125
48,Brook Lopez,348,-0.146,-0.114,-0.199,-0.094
157,Ivica Zubac,260,-0.155,-0.112,-0.215,-0.094
132,Evan Mobley,433,-0.137,-0.111,-0.184,-0.090
373,Robert Williams III,219,-0.161,-0.111,-0.226,-0.095


In [33]:
filenames = {
    "Overall": "defend_overall.parquet",
    "Less Than 6Ft": "defend_less_than_6ft.parquet",
    "3 Pointers": "defend_3_pointers.parquet",
}

for category, filename in filenames.items():
    df = defended_tables[category]

    assert df["CLOSE_DEF_PERSON_ID"].is_unique

    path = RAW_DATA_DIR / filename
    df.to_parquet(path, index=False)

    # Verify that the saved file can be read.
    saved = pd.read_parquet(path)

    assert saved.shape == df.shape
    assert saved.columns.equals(df.columns)

    print(f"Saved: {path} ({len(saved)} rows)")

Saved: C:\Users\aabre\Documents\data-science-projects\nba-player-archetype-analysis\data\raw\2025_26\defend_overall.parquet (580 rows)
Saved: C:\Users\aabre\Documents\data-science-projects\nba-player-archetype-analysis\data\raw\2025_26\defend_less_than_6ft.parquet (576 rows)
Saved: C:\Users\aabre\Documents\data-science-projects\nba-player-archetype-analysis\data\raw\2025_26\defend_3_pointers.parquet (575 rows)


In [34]:

# Compare three shrinkage strengths for rim defense.
rim = defensive_performance.copy()

n = rim["RIM_DFGA"]
raw_diff = (
    rim["RIM_DFGM"] / n
    - rim["RIM_NORMAL_FG_PCT"]
)

for k in [50, 100, 200]:
    rim[f"RIM_SHRUNK_K{k}"] = (
        n / (n + k)
    ) * raw_diff

    rim[f"RANK_K{k}"] = rim[
        f"RIM_SHRUNK_K{k}"
    ].rank(
        ascending=True,
        method="min",
    )

# Examine how player rankings change.
display(
    rim[
        [
            "RIM_SHRUNK_K50",
            "RIM_SHRUNK_K100",
            "RIM_SHRUNK_K200",
        ]
    ].corr(method="spearman").round(3)
)

# Identify players whose rankings are most sensitive.
rim["RANK_RANGE"] = (
    rim[["RANK_K50", "RANK_K100", "RANK_K200"]].max(axis=1)
    - rim[["RANK_K50", "RANK_K100", "RANK_K200"]].min(axis=1)
)

display(
    rim.nlargest(15, "RANK_RANGE")[
        [
            "PLAYER_NAME",
            "RIM_DFGA",
            "RANK_K50",
            "RANK_K100",
            "RANK_K200",
            "RANK_RANGE",
        ]
    ]
)


,RIM_SHRUNK_K50,RIM_SHRUNK_K100,RIM_SHRUNK_K200
RIM_SHRUNK_K50,1.000,0.997,0.991
RIM_SHRUNK_K100,0.997,1.000,0.998
RIM_SHRUNK_K200,0.991,0.998,1.000


,PLAYER_NAME,RIM_DFGA,RANK_K50,RANK_K100,RANK_K200,RANK_RANGE
306,Matisse Thybulle,20,406.0,360.0,334.0,72.0
82,Cormac Ryan,16,321.0,281.0,263.0,58.0
308,Max Strus,32,358.0,327.0,301.0,57.0
224,Jordan McLaughlin,23,341.0,310.0,285.0,56.0
298,Malaki Branham,20,319.0,284.0,267.0,52.0
333,Nick Smith Jr.,34,353.0,326.0,302.0,51.0
47,Bronny James,39,396.0,373.0,346.0,50.0
168,Jake LaRavia,278,314.0,336.0,364.0,50.0
216,John Poulakidas,32,369.0,342.0,320.0,49.0
127,Dyson Daniels,270,338.0,362.0,384.0,46.0


In [35]:

# Restrict comparisons to players with reasonable rim exposure.
rim_qualified = rim[
    rim["RIM_DFGA"] >= 100
].copy()

print("Qualified players:", len(rim_qualified))

top_20 = {}

for k in [50, 100, 200]:
    top_20[k] = set(
        rim_qualified.nsmallest(
            20, f"RIM_SHRUNK_K{k}"
        )["PLAYER_ID"]
    )

for a, b in [(50, 100), (100, 200), (50, 200)]:
    overlap = len(top_20[a] & top_20[b])

    print(
        f"k={a} vs k={b}: "
        f"{overlap}/20 players shared"
    )

# Show the top players using the provisional baseline.
display(
    rim_qualified.nsmallest(
        15, "RIM_SHRUNK_K100"
    )[
        [
            "PLAYER_NAME",
            "RIM_DFGA",
            "RIM_FG_DIFF",
            "RIM_SHRUNK_K100",
            "RIM_DIFF_CI_LOW",
            "RIM_DIFF_CI_HIGH",
        ]
    ].round(3)
)


Qualified players: 307
k=50 vs k=100: 19/20 players shared
k=100 vs k=200: 19/20 players shared
k=50 vs k=200: 18/20 players shared


,PLAYER_NAME,RIM_DFGA,RIM_FG_DIFF,RIM_SHRUNK_K100,RIM_DIFF_CI_LOW,RIM_DIFF_CI_HIGH
156,Isaiah Stewart,255,-0.228,-0.164,-0.286,-0.167
70,Chet Holmgren,433,-0.183,-0.149,-0.230,-0.136
434,Victor Wembanyama,411,-0.172,-0.138,-0.220,-0.124
73,Clint Capela,201,-0.182,-0.121,-0.249,-0.112
381,Ryan Kalkbrenner,372,-0.147,-0.116,-0.198,-0.096
266,Khaman Maluach,116,-0.215,-0.116,-0.302,-0.125
48,Brook Lopez,348,-0.146,-0.114,-0.199,-0.094
157,Ivica Zubac,260,-0.155,-0.112,-0.215,-0.094
132,Evan Mobley,433,-0.137,-0.111,-0.184,-0.090
373,Robert Williams III,219,-0.161,-0.111,-0.226,-0.095


In [36]:
reb = defensive_performance.copy()

# Calculate conversion using the original counts.
n = reb["DREB_CHANCES"].astype(float)
made = reb["DREB"].astype(float)

assert ((n > 0) & (made >= 0) & (made <= n)).all()

reb["REB_RATE"] = made / n

# Verify against the NBA's published percentage.
print(
    "Largest difference from published rate:",
    (reb["REB_RATE"] - reb["DREB_CHANCE_PCT"]).abs().max()
)

# Pooled rate across eligible players, weighted by chances.
baseline = made.sum() / n.sum()

print("Pooled defensive rebound conversion rate:",
      round(baseline, 4))

# Experimental shrinkage toward the pooled baseline.
k = 100

reb["REB_RATE_SHRUNK"] = (
    made + k * baseline
) / (n + k)

# Approximate 95% Wilson intervals.
z = 1.96
p = reb["REB_RATE"]

denominator = 1 + z**2 / n

center = (
    p + z**2 / (2 * n)
) / denominator

margin = (
    z * np.sqrt(
        p * (1 - p) / n
        + z**2 / (4 * n**2)
    )
) / denominator

reb["REB_CI_LOW"] = center - margin
reb["REB_CI_HIGH"] = center + margin

print("\nPlayers with >= 100 chances:",
      n.ge(100).sum())

print("Players with >= 200 chances:",
      n.ge(200).sum())

print("\nHighest shrunk conversion rates:")

display(
    reb[reb["DREB_CHANCES"] >= 100]
    .nlargest(15, "REB_RATE_SHRUNK")[
        [
            "PLAYER_NAME",
            "DREB_CHANCES",
            "REB_RATE",
            "REB_RATE_SHRUNK",
            "REB_CI_LOW",
            "REB_CI_HIGH",
        ]
    ].round(3)
)


Largest difference from published rate: 0.0004999999999999449
Pooled defensive rebound conversion rate: 0.739

Players with >= 100 chances: 362
Players with >= 200 chances: 232

Highest shrunk conversion rates:


,PLAYER_NAME,DREB_CHANCES,REB_RATE,REB_RATE_SHRUNK,REB_CI_LOW,REB_CI_HIGH
245,Karl-Anthony Towns,741,0.885,0.868,0.860,0.906
353,Paolo Banchero,591,0.878,0.858,0.849,0.902
90,Danny Wolf,249,0.900,0.854,0.856,0.931
434,Victor Wembanyama,706,0.858,0.844,0.831,0.882
373,Robert Williams III,314,0.873,0.840,0.831,0.905
139,Giannis Antetokounmpo,292,0.870,0.836,0.826,0.904
242,Jusuf Nurkić,371,0.860,0.834,0.821,0.891
263,Kevin Porter Jr.,186,0.882,0.832,0.827,0.921
313,Michael Porter Jr.,349,0.854,0.828,0.813,0.887
29,Bam Adebayo,695,0.839,0.826,0.810,0.864


In [37]:
# Add existing positional matchup shares.
reb = reb.merge(
    time_shares[["G", "F", "C"]],
    left_on="PLAYER_ID",
    right_index=True,
    how="left",
    validate="one_to_one",
)

assert reb[["G", "F", "C"]].notna().all().all()

# Check relationships with defensive roles.
columns = [
    "REB_RATE",
    "REB_RATE_SHRUNK",
    "DREB_CHANCES",
    "DREB_CONTEST_PCT",
    "G",
    "F",
    "C",
]

display(
    reb[columns].corr().round(3)
)


,REB_RATE,REB_RATE_SHRUNK,DREB_CHANCES,DREB_CONTEST_PCT,G,F,C
REB_RATE,1.000,0.955,0.333,0.021,-0.104,0.172,0.027
REB_RATE_SHRUNK,0.955,1.000,0.344,0.003,-0.119,0.210,0.023
DREB_CHANCES,0.333,0.344,1.000,0.279,-0.402,0.246,0.352
DREB_CONTEST_PCT,0.021,0.003,0.279,1.000,-0.756,0.392,0.703
G,-0.104,-0.119,-0.402,-0.756,1.000,-0.602,-0.881
F,0.172,0.210,0.246,0.392,-0.602,1.000,0.154
C,0.027,0.023,0.352,0.703,-0.881,0.154,1.000


In [38]:
# Choose a player to evaluate.
anchor_name = "Karl-Anthony Towns"

# Require at least 100 defensive rebound chances.
qualified = reb[
    reb["DREB_CHANCES"] >= 100
].copy()

anchor = qualified.loc[
    qualified["PLAYER_NAME"] == anchor_name
].iloc[0]

role_cols = ["G", "F", "C"]

# Difference in positional matchup distributions.
anchor_shares = anchor[role_cols].astype(float)

qualified["ROLE_DISTANCE"] = (
    qualified[role_cols]
    .sub(anchor_shares)
    .abs()
    .sum(axis=1) / 2
)

# Exclude the selected player and identify nearest roles.
similar_roles = (
    qualified[
        qualified["PLAYER_ID"] != anchor["PLAYER_ID"]
    ]
    .nsmallest(15, "ROLE_DISTANCE")
    .copy()
)

print(f"Selected player: {anchor_name}")

print(
    "Player's shrunk rebound conversion:",
    round(anchor["REB_RATE_SHRUNK"], 3)
)

print(
    "Pooled conversion among 15 similar roles:",
    round(
        similar_roles["DREB"].sum()
        / similar_roles["DREB_CHANCES"].sum(),
        3
    )
)

display(
    similar_roles[
        [
            "PLAYER_NAME",
            "ROLE_DISTANCE",
            "G", "F", "C",
            "DREB_CHANCES",
            "REB_RATE",
            "REB_RATE_SHRUNK",
            "DREB_CONTEST_PCT",
        ]
    ].round(3)
)


Selected player: Karl-Anthony Towns
Player's shrunk rebound conversion: 0.868
Pooled conversion among 15 similar roles: 0.717


,PLAYER_NAME,ROLE_DISTANCE,G,F,C,DREB_CHANCES,REB_RATE,REB_RATE_SHRUNK,DREB_CONTEST_PCT
7,Adem Bona,0.008,0.196,0.363,0.442,289,0.619,0.650,0.408
348,Onyeka Okongwu,0.011,0.182,0.366,0.452,561,0.752,0.750,0.313
237,Julian Reese,0.012,0.177,0.371,0.452,115,0.713,0.725,0.317
169,Jakob Poeltl,0.013,0.196,0.358,0.447,304,0.688,0.700,0.464
310,Maxime Raynaud,0.019,0.207,0.365,0.428,515,0.804,0.793,0.304
156,Isaiah Stewart,0.019,0.201,0.377,0.422,296,0.655,0.677,0.371
251,Kel'el Ware,0.023,0.211,0.350,0.439,612,0.779,0.774,0.365
304,Marvin Bagley III,0.026,0.171,0.363,0.467,278,0.777,0.767,0.398
73,Clint Capela,0.026,0.171,0.397,0.432,258,0.717,0.723,0.422
442,Yanic Konan Niederhäuser,0.026,0.162,0.395,0.443,103,0.777,0.758,0.275


In [39]:
# Start with our existing rebounding and matchup data.
cols = [
    "PLAYER_ID", "PLAYER_NAME",
    "DREB", "DREB_CHANCES", "DREB_CHANCE_DEFER",
    "DREB_CONTEST_PCT", "G", "F", "C",
]

reb_audit = reb[cols].copy()

# Calculate opportunity and conversion statistics.
reb_audit["RAW_RATE"] = (
    reb_audit["DREB"] / reb_audit["DREB_CHANCES"]
)

reb_audit["DEFER_SHARE"] = (
    reb_audit["DREB_CHANCE_DEFER"]
    / reb_audit["DREB_CHANCES"]
)

reb_audit["NONDEFER_CHANCES"] = (
    reb_audit["DREB_CHANCES"]
    - reb_audit["DREB_CHANCE_DEFER"]
)

reb_audit["ADJUSTED_RATIO"] = (
    reb_audit["DREB"]
    / reb_audit["NONDEFER_CHANCES"].replace(0, np.nan)
)

print("Deferred chance share distribution:")
display(reb_audit["DEFER_SHARE"].describe().round(3))

print(
    "Players with adjusted ratios above 100%:",
    reb_audit["ADJUSTED_RATIO"].gt(1).sum()
)

print("\nRelationships with deferral:")
display(
    reb_audit[
        [
            "RAW_RATE",
            "DEFER_SHARE",
            "DREB_CONTEST_PCT",
            "G", "F", "C",
        ]
    ].corr().round(3)
)

# Rebuild Towns' positional-assignment peers.
qualified = reb_audit[
    reb_audit["DREB_CHANCES"] >= 100
].copy()

towns = qualified.loc[
    qualified["PLAYER_NAME"] == "Karl-Anthony Towns"
].iloc[0]

role_cols = ["G", "F", "C"]

qualified["ROLE_DISTANCE"] = (
    qualified[role_cols]
    .sub(towns[role_cols].astype(float))
    .abs().sum(axis=1) / 2
)

peers = (
    qualified[
        qualified["PLAYER_ID"] != towns["PLAYER_ID"]
    ]
    .nsmallest(15, "ROLE_DISTANCE")
    .copy()
)

# Compare like-for-like pooled raw outcomes.
peer_rebounds = peers["DREB"].sum()
peer_chances = peers["DREB_CHANCES"].sum()

summary = pd.DataFrame([
    {
        "GROUP": "Karl-Anthony Towns",
        "RAW_CONVERSION": towns["RAW_RATE"],
        "DEFER_SHARE": towns["DEFER_SHARE"],
        "ADJUSTED_RATIO": towns["ADJUSTED_RATIO"],
        "CONTESTED_REBOUND_SHARE": towns["DREB_CONTEST_PCT"],
    },
    {
        "GROUP": "15 assignment-matched peers",
        "RAW_CONVERSION": peer_rebounds / peer_chances,
        "DEFER_SHARE": (
            peers["DREB_CHANCE_DEFER"].sum()
            / peer_chances
        ),
        "ADJUSTED_RATIO": (
            peer_rebounds
            / peers["NONDEFER_CHANCES"].sum()
        ),
        "CONTESTED_REBOUND_SHARE": (
            (peers["DREB_CONTEST_PCT"] * peers["DREB"]).sum()
            / peer_rebounds
        ),
    },
])

display(summary.round(3))

Deferred chance share distribution:


count    450.000
mean       0.185
std        0.049
min        0.043
25%        0.151
50%        0.181
75%        0.212
max        0.346
Name: DEFER_SHARE, dtype: float64

Players with adjusted ratios above 100%: 24

Relationships with deferral:


,RAW_RATE,DEFER_SHARE,DREB_CONTEST_PCT,G,F,C
RAW_RATE,1.000,-0.558,0.021,-0.104,0.172,0.027
DEFER_SHARE,-0.558,1.000,0.259,-0.275,0.078,0.294
DREB_CONTEST_PCT,0.021,0.259,1.000,-0.756,0.392,0.703
G,-0.104,-0.275,-0.756,1.000,-0.602,-0.881
F,0.172,0.078,0.392,-0.602,1.000,0.154
C,0.027,0.294,0.703,-0.881,0.154,1.000


,GROUP,RAW_CONVERSION,DEFER_SHARE,ADJUSTED_RATIO,CONTESTED_REBOUND_SHARE
0,Karl-Anthony Towns,0.885,0.151,1.043,0.290
1,15 assignment-matched peers,0.717,0.211,0.908,0.356


In [40]:
from scipy.optimize import minimize
from scipy.special import betaln

# Player-level rebound successes and opportunities.
y = reb_audit["DREB"].to_numpy(dtype=float)
n = reb_audit["DREB_CHANCES"].to_numpy(dtype=float)

assert np.all((n > 0) & (y >= 0) & (y <= n))

# Initial values for the optimization.
pooled_rate = y.sum() / n.sum()
starting_k = 100

initial_alpha = pooled_rate * starting_k
initial_beta = (1 - pooled_rate) * starting_k


def negative_log_likelihood(log_parameters):
    alpha, beta = np.exp(log_parameters)

    # Beta-binomial likelihood, omitting terms
    # constant with respect to alpha and beta.
    return -np.sum(
        betaln(y + alpha, n - y + beta)
        - betaln(alpha, beta)
    )


fit = minimize(
    negative_log_likelihood,
    x0=np.log([initial_alpha, initial_beta]),
    method="L-BFGS-B",
    bounds=[(-5, 12), (-5, 12)],
)

if not fit.success:
    raise RuntimeError(fit.message)

alpha, beta = np.exp(fit.x)

eb_mean = alpha / (alpha + beta)
eb_k = alpha + beta

print("Estimated baseline:", round(eb_mean, 4))
print("Estimated shrinkage strength:", round(eb_k, 2))

# Empirical Bayes estimates.
reb_audit["REB_RATE_EB"] = (
    reb_audit["DREB"] + alpha
) / (
    reb_audit["DREB_CHANCES"] + alpha + beta
)

# Our previous heuristic method for comparison.
reb_audit["REB_RATE_K100"] = (
    reb_audit["DREB"] + 100 * pooled_rate
) / (
    reb_audit["DREB_CHANCES"] + 100
)

print("\nCorrelation between methods:")
print(
    reb_audit["REB_RATE_EB"].corr(
        reb_audit["REB_RATE_K100"],
        method="spearman",
    )
)

# Compare top-20 players with at least 100 opportunities.
qualified_eb = reb_audit[
    reb_audit["DREB_CHANCES"] >= 100
].copy()

top_eb = set(
    qualified_eb.nlargest(
        20, "REB_RATE_EB"
    )["PLAYER_ID"]
)

top_k100 = set(
    qualified_eb.nlargest(
        20, "REB_RATE_K100"
    )["PLAYER_ID"]
)

print("Top-20 overlap:", len(top_eb & top_k100), "/20")

display(
    qualified_eb.nlargest(
        15, "REB_RATE_EB"
    )[
        [
            "PLAYER_NAME",
            "DREB_CHANCES",
            "RAW_RATE",
            "REB_RATE_K100",
            "REB_RATE_EB",
        ]
    ].round(3)
)


Estimated baseline: 0.7263
Estimated shrinkage strength: 47.95

Correlation between methods:
0.9970728403555604
Top-20 overlap: 18 /20


,PLAYER_NAME,DREB_CHANCES,RAW_RATE,REB_RATE_K100,REB_RATE_EB
245,Karl-Anthony Towns,741,0.885,0.868,0.876
90,Danny Wolf,249,0.900,0.854,0.872
353,Paolo Banchero,591,0.878,0.858,0.867
373,Robert Williams III,314,0.873,0.840,0.853
434,Victor Wembanyama,706,0.858,0.844,0.850
263,Kevin Porter Jr.,186,0.882,0.832,0.850
139,Giannis Antetokounmpo,292,0.870,0.836,0.850
242,Jusuf Nurkić,371,0.860,0.834,0.845
313,Michael Porter Jr.,349,0.854,0.828,0.838
318,Mitchell Robinson,320,0.853,0.826,0.837


In [42]:
activity_cols = [
    "DEFENSIVE_DISRUPTION_SCORE",
    "PERIMETER_DEFENSE_SCORE",
    "INTERIOR_PRESENCE_SCORE",
]

role_cols = ["G", "F", "C"]

# Combine assignments and activity features.
defensive_roles = reb_audit[
    ["PLAYER_ID", "PLAYER_NAME", *role_cols]
].merge(
    defense_experiment[
        ["PLAYER_ID", *activity_cols]
    ],
    on="PLAYER_ID",
    validate="one_to_one",
)

assert len(defensive_roles) == 450
assert defensive_roles[role_cols + activity_cols].notna().all().all()


def compare_defensive_roles(
    players,
    player_name,
    candidate_count=50,
    top_n=10,
):
    """Compare assignment-only and assignment-plus-activity peers."""

    anchor = players.loc[
        players["PLAYER_NAME"] == player_name
    ].iloc[0]

    candidates = players.loc[
        players["PLAYER_ID"] != anchor["PLAYER_ID"]
    ].copy()

    # Assignment distance: total variation distance.
    candidates["ASSIGNMENT_DISTANCE"] = (
        candidates[role_cols]
        .sub(anchor[role_cols].astype(float))
        .abs()
        .sum(axis=1) / 2
    )

    # Method A: closest positional assignments.
    assignment_only = candidates.nsmallest(
        top_n, "ASSIGNMENT_DISTANCE"
    )

    # Method B: take the closest 50 assignments.
    pool = candidates.nsmallest(
        candidate_count, "ASSIGNMENT_DISTANCE"
    ).copy()

    # Rank that candidate pool by defensive activity.
    pool["ACTIVITY_DISTANCE"] = np.sqrt(
        np.mean(
            (
                pool[activity_cols].to_numpy(dtype=float)
                - anchor[activity_cols].to_numpy(dtype=float)
            ) ** 2,
            axis=1,
        )
    )

    assignment_activity = pool.nsmallest(
        top_n, "ACTIVITY_DISTANCE"
    )

    return assignment_only, assignment_activity

examples = [
    "Karl-Anthony Towns",
    "Alex Caruso",
    "OG Anunoby",
]

for player_name in examples:

    assignment_only, combined = compare_defensive_roles(
        defensive_roles,
        player_name,
    )

    print(f"\n{player_name}")

    print("Assignment-only peers:")
    display(
        assignment_only[
            [
                "PLAYER_NAME",
                "ASSIGNMENT_DISTANCE",
            ]
        ].round(3)
    )

    print("Assignment + activity peers:")
    display(
        combined[
            [
                "PLAYER_NAME",
                "ASSIGNMENT_DISTANCE",
                "ACTIVITY_DISTANCE",
            ]
        ].round(3)
    )

    overlap = len(
        set(assignment_only["PLAYER_ID"])
        & set(combined["PLAYER_ID"])
    )

    print("Top-10 overlap:", overlap, "/10")


Karl-Anthony Towns
Assignment-only peers:


,PLAYER_NAME,ASSIGNMENT_DISTANCE
7,Adem Bona,0.008
348,Onyeka Okongwu,0.011
237,Julian Reese,0.012
169,Jakob Poeltl,0.013
310,Maxime Raynaud,0.019
156,Isaiah Stewart,0.019
251,Kel'el Ware,0.023
304,Marvin Bagley III,0.026
73,Clint Capela,0.026
442,Yanic Konan Niederhäuser,0.026


Assignment + activity peers:


,PLAYER_NAME,ASSIGNMENT_DISTANCE,ACTIVITY_DISTANCE
7,Adem Bona,0.008,0.067
282,Lachlan Olbrich,0.067,0.120
348,Onyeka Okongwu,0.011,0.254
422,Tristan Vukcevic,0.038,0.301
336,Nikola Jokić,0.034,0.313
331,Nic Claxton,0.051,0.344
325,Myles Turner,0.057,0.360
21,Anthony Davis,0.068,0.368
275,Kristaps Porziņģis,0.038,0.389
324,Moussa Diabaté,0.030,0.431


Top-10 overlap: 2 /10

Alex Caruso
Assignment-only peers:


,PLAYER_NAME,ASSIGNMENT_DISTANCE
65,Carter Bryant,0.005
215,John Konchar,0.007
394,Sidy Cissoko,0.017
274,Kris Murray,0.019
369,Rasheer Fleming,0.024
328,Naji Marshall,0.028
36,Blake Hinson,0.028
155,Isaiah Livers,0.030
234,Jrue Holiday,0.031
379,Russell Westbrook,0.031


... (3 outputs left out)

,PLAYER_NAME,ASSIGNMENT_DISTANCE
202,Jayson Tatum,0.019
412,Tobias Harris,0.019
288,Leonard Miller,0.023
214,John Collins,0.024
267,Khris Middleton,0.028
316,Miles Bridges,0.030
175,Jalen Slawson,0.034
449,Zion Williamson,0.034
260,Kevin Durant,0.036
173,Jalen Johnson,0.037


Assignment + activity peers:


,PLAYER_NAME,ASSIGNMENT_DISTANCE,ACTIVITY_DISTANCE
187,Jarace Walker,0.082,0.265
161,Jabari Walker,0.051,0.384
357,Patrick Williams,0.039,0.401
256,Kenrich Williams,0.064,0.439
239,Julius Randle,0.082,0.488
376,Royce O'Neale,0.041,0.522
410,Tidjane Salaün,0.043,0.535
178,Jalen Williams,0.069,0.546
284,Larry Nance Jr.,0.082,0.547
313,Michael Porter Jr.,0.085,0.559


Top-10 overlap: 0 /10
